In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


In [2]:
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


In [3]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv')

print(train.shape)
print(test.shape)

train.head()

/tmp/ipykernel_16/2019506285.py:3: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv')


(138701, 50)
(15000, 49)


,TransactionID,TargetValue,AssetID,ProductConfigID,DataOriginCode,VendorPartnerID,ManufactureYear,OperationalHoursMeter,UtilizationTier,TransactionDate,...,col20,col21,col22,col23,col24,col25,col27,col28,col29,col30
0,1139309,57000.0,117665,88,ACH138,GTX,1997,4640.0,Low,2005-03-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
1,1139316,26500.0,1001282,4616,ACH138,GTX,2005,508.0,Low,2009-12-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1139317,21000.0,772709,1948,ACH138,GTX,1994,11540.0,High,2005-08-26,...,Steel,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN
3,1139322,27000.0,902010,3550,ACH138,GTX,2002,4883.0,High,2006-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1139333,21500.0,1036259,36014,ACH138,GTX,2009,302.0,Low,2010-08-27,...,Rubber,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN


In [4]:
import numpy as np
import pandas as pd

def engineer_features(df, is_train=True):
    # Create a copy to avoid SettingWithCopy warnings
    df_feat = df.copy()
    
    # 1. Clean the 1001 Manufacture Year Trap
    # Replace years <= 1950 with NaN so models treat them as missing data
    df_feat['ManufactureYear'] = df_feat['ManufactureYear'].apply(lambda x: np.nan if x <= 1950 else x)
    
    # 2. Date Feature Engineering
    df_feat['TransactionDate'] = pd.to_datetime(df_feat['TransactionDate'])
    df_feat['TransactionYear'] = df_feat['TransactionDate'].dt.year
    df_feat['TransactionMonth'] = df_feat['TransactionDate'].dt.month
    df_feat['TransactionDay'] = df_feat['TransactionDate'].dt.day
    df_feat['TransactionDayOfWeek'] = df_feat['TransactionDate'].dt.dayofweek
    
    # 3. Age of Equipment at Sale (The single most important engineered feature)
    df_feat['EquipmentAgeAtSale'] = df_feat['TransactionYear'] - df_feat['ManufactureYear']
    
    # 4. Handle Operational Hours Outliers
    # Cap hours at the 99th percentile (approx 20,000 to 30,000 hours usually for heavy machinery)
    # We use a fixed cap based on train set or a reasonable maximum to keep train/test aligned
    hour_cap = 30000 
    df_feat['OperationalHoursMeter'] = df_feat['OperationalHoursMeter'].clip(upper=hour_cap)
    
    # Create an indicator if hours were missing
    df_feat['Hours_Is_Missing'] = df_feat['OperationalHoursMeter'].isnull().astype(int)
    
    # 5. Drop the ultra-empty columns (>99% missing values)
    cols_to_drop = ['col18', 'col19']
    df_feat = df_feat.drop(columns=[col for col in cols_to_drop if col in df_feat.columns])
    
    # 6. Fill missing values for object/categorical columns with a specific string
    # This ensures "NaN" becomes a visible class for tree models
    cat_cols = df_feat.select_dtypes(include=['object']).columns
    df_feat[cat_cols] = df_feat[cat_cols].fillna('Missing_Value')
    
    # Convert all string columns explicitly to 'category' type for LightGBM/CatBoost
    for col in cat_cols:
        df_feat[col] = df_feat[col].astype('category')
        
    # 7. Drop ID and raw date columns that shouldn't be used directly as predictive features
    cols_to_remove = ['TransactionID', 'TransactionDate']
    df_feat = df_feat.drop(columns=[col for col in cols_to_remove if col in df_feat.columns])
    
    # 8. Target Transformation (Only for Training set)
    if is_train:
        # We drop AssetID from training features to prevent overfitting on specific assets
        if 'AssetID' in df_feat.columns:
            df_feat = df_feat.drop(columns=['AssetID'])
        
        # Apply Log transformation to Target Value to directly optimize for RMSLE
        y = np.log1p(df_feat['TargetValue'])
        X = df_feat.drop(columns=['TargetValue'])
        return X, y
    else:
        return df_feat

# Apply the pipeline
X_train, y_train = engineer_features(train, is_train=True)
X_test = engineer_features(test, is_train=False)

print("--- Data Engineering Complete ---")
print(f"Engineered Train Shape: {X_train.shape}")
print(f"Engineered Test Shape:  {X_test.shape}")

--- Data Engineering Complete ---
Engineered Train Shape: (138701, 50)
Engineered Test Shape:  (15000, 51)


In [5]:
# 1. Fix the negative age anomaly in both sets
X_train['EquipmentAgeAtSale'] = X_train['EquipmentAgeAtSale'].apply(lambda x: np.nan if x < 0 else x)
X_test['EquipmentAgeAtSale'] = X_test['EquipmentAgeAtSale'].apply(lambda x: np.nan if x < 0 else x)

# 2. Drop AssetID from test features so columns match X_train exactly
if 'AssetID' in X_test.columns:
    X_test = X_test.drop(columns=['AssetID'])

print("--- Final Validation ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Are features perfectly identical?: {list(X_train.columns) == list(X_test.columns)}")

--- Final Validation ---
X_train shape: (138701, 50)
X_test shape:  (15000, 50)
Are features perfectly identical?: True


In [6]:
print("--- 1. Nonlinear Depreciation Check ---")
# Heavy equipment doesn't depreciate linearly. Let's see the median price by age brackets
train['Age_Group'] = pd.qcut(X_train['EquipmentAgeAtSale'].dropna(), q=5, labels=['Very New', 'New', 'Mid-Age', 'Old', 'Very Old'])
print(train.groupby('Age_Group', observed=False)['TargetValue'].median())

print("\n--- 2. Interaction Feature Diagnostic ---")
# Does having high hours matter more for older machines vs newer machines?
train['Age'] = X_train['EquipmentAgeAtSale']
train['Hours'] = X_train['OperationalHoursMeter']
print("Median price of New Machine with High Hours vs Low Hours:")
print(train[train['Age'] <= 5].groupby(train['Hours'] > 5000, observed=False)['TargetValue'].median())
print("\nMedian price of Old Machine with High Hours vs Low Hours:")
print(train[train['Age'] > 15].groupby(train['Hours'] > 5000, observed=False)['TargetValue'].median())

print("\n--- 3. Time-Based Distribution Shift Check ---")
# Are we predicting the future? Let's check the transaction years in Train vs Test
print("Train Transaction Years distribution:")
print(X_train['TransactionYear'].value_counts().sort_index())
print("\nTest Transaction Years distribution:")
print(X_test['TransactionYear'].value_counts().sort_index())

--- 1. Nonlinear Depreciation Check ---
Age_Group
Very New    56000.0
New         43000.0
Mid-Age     36000.0
Old         30000.0
Very Old    24000.0
Name: TargetValue, dtype: float64

--- 2. Interaction Feature Diagnostic ---
Median price of New Machine with High Hours vs Low Hours:
Hours
False    51000.0
True     63000.0
Name: TargetValue, dtype: float64

Median price of Old Machine with High Hours vs Low Hours:
Hours
False    22500.0
True     25000.0
Name: TargetValue, dtype: float64

--- 3. Time-Based Distribution Shift Check ---
Train Transaction Years distribution:
TransactionYear
1990      577
1991      518
1992      589
1993      560
1994      657
1995     1338
1996     1416
1997     1678
1998     1949
1999     2910
2000     2735
2001     3814
2002     3833
2003     3911
2004     3452
2005     5303
2006     5746
2007     6142
2008    11985
2009    16809
2010    21333
2011    17880
2012    18472
2013     5094
Name: count, dtype: int64

Test Transaction Years distribution:
Transa

In [7]:
# 1. Calculate the Expected Hours per Year to capture real machine strain
# Adding 1 to avoid division by zero for brand new machines
X_train['Hours_Per_Year'] = X_train['OperationalHoursMeter'] / (X_train['EquipmentAgeAtSale'] + 1)
X_test['Hours_Per_Year'] = X_test['OperationalHoursMeter'] / (X_test['EquipmentAgeAtSale'] + 1)

# 2. Group Target Encoding Framework (Crucial to break the Hours Paradox)
# Calculate the mean operational hours per machine base class to understand "What is normal usage for this specific model?"
class_mean_hours = train.groupby('Spec_BaseClass', observed=False)['OperationalHoursMeter'].mean().to_dict()

# Map the baseline expectations back to train and test
X_train['BaseClass_Mean_Hours'] = train['Spec_BaseClass'].map(class_mean_hours).astype(float)
X_test['BaseClass_Mean_Hours'] = test['Spec_BaseClass'].map(class_mean_hours).astype(float)

# 3. Create Relative Usage Feature: Is this specific unit overworked or underworked compared to its peers?
X_train['Relative_Usage_Index'] = X_train['OperationalHoursMeter'] / (X_train['BaseClass_Mean_Hours'] + 1)
X_test['Relative_Usage_Index'] = X_test['OperationalHoursMeter'] / (X_test['BaseClass_Mean_Hours'] + 1)

# 4. Handle the exponential decay by generating a log-age feature
X_train['Log_Equipment_Age'] = np.log1p(X_train['EquipmentAgeAtSale'])
X_test['Log_Equipment_Age'] = np.log1p(X_test['EquipmentAgeAtSale'])

print("--- Advanced Domain Engineering Complete ---")
print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape:  {X_test.shape}")

--- Advanced Domain Engineering Complete ---
X_train Shape: (138701, 54)
X_test Shape:  (15000, 54)


In [8]:
# import lightgbm as lgb
# from sklearn.model_selection import KFold
# from sklearn.metrics import root_mean_squared_error # Available in newer scikit-learn
# # Alternative if sklearn is older: from sklearn.metrics import mean_squared_error

# def rmsle(y_true, y_pred):
#     # Since y_train is already log1p transformed, a standard RMSE calculation on it is mathematically equivalent to RMSLE
#     return root_mean_squared_error(y_true, y_pred)

# # Initialize K-Fold
# kf = KFold(n_splits=5, shuffle=True, random_state=42)
# oof_predictions = np.zeros(len(X_train))
# test_predictions = np.zeros(len(X_test))

# print("--- Starting 5-Fold Cross-Validation Process ---")

# for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
#     X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
#     X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
#     # Initialize LightGBM Regressor optimized for Kaggle speed/performance
#     model = lgb.LGBMRegressor(
#         objective='regression',
#         metric='rmse',
#         n_estimators=1500,
#         learning_rate=0.05,
#         num_leaves=63,
#         random_state=42,
#         n_jobs=-1
#     )
    
#     # Train model with early stopping to prevent overfitting
#     model.fit(
#         X_tr, y_tr,
#         eval_set=[(X_va, y_va)],
#         callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
#     )
    
#     # Predict on validation set
#     oof_predictions[val_idx] = model.predict(X_va)
    
#     # Predict on test set (average predictions across all folds)
#     test_predictions += model.predict(X_test) / kf.n_splits
    
#     fold_score = rmsle(y_va, oof_predictions[val_idx])
#     print(f"Fold {fold + 1} RMSLE: {fold_score:.4f}")

# cv_score = rmsle(y_train, oof_predictions)
# print(f"\n======================================")
# print(f"Overall Out-of-Fold CV RMSLE Score: {cv_score:.4f}")
# print(f"======================================")

In [9]:
# # Convert predictions back to original scale
# final_predictions = np.expm1(test_predictions)

# # Optional: Remove any negative predictions (rare but possible)
# final_predictions = np.clip(final_predictions, 0, None)

# # Create submission DataFrame
# submission = pd.DataFrame({
#     "TransactionID": test["TransactionID"],
#     "TargetValue": final_predictions
# })

# # Save submission file
# submission.to_csv("submission.csv", index=False)

# print("Submission file created successfully!")
# submission.head()

****so here the score crosses cutoff now changes beyond****

In [10]:
# import lightgbm as lgb
# from sklearn.model_selection import KFold
# from sklearn.metrics import root_mean_squared_error

# # Initialize K-Fold identically to keep comparison clean
# kf = KFold(n_splits=5, shuffle=True, random_state=42)
# oof_predictions = np.zeros(len(X_train))
# test_predictions = np.zeros(len(X_test))

# print("--- Starting Hyperparameter-Optimized 5-Fold CV ---")

# for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
#     X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
#     X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
#     # Tuned explicitly for high-cardinality Kaggle competitions
#     model = lgb.LGBMRegressor(
#         objective='regression',
#         metric='rmse',
#         n_estimators=3000,         # Increased estimators for deeper learning
#         learning_rate=0.03,        # Lowered learning rate for finer convergence
#         num_leaves=127,            # Increased from 63 to capture complex structural interactions
#         max_bin=1000,              # CRITICAL: Increases numerical precision
#         max_cat_to_onehot=10,
#         cat_l2=20,                 # Regularization for high-cardinality categories
#         cat_smooth=25,             # Reduces noise in rare equipment models
#         colsample_bytree=0.7,      # Feature subsampling prevents dominant features from hogging splits
#         subsample=0.8,             # Row subsampling adds bagging variance
#         random_state=42,
#         n_jobs=-1
#     )
    
#     model.fit(
#         X_tr, y_tr,
#         eval_set=[(X_va, y_va)],
#         callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)] # Expanded patience
#     )
    
#     oof_predictions[val_idx] = model.predict(X_va)
#     test_predictions += model.predict(X_test) / kf.n_splits
    
#     fold_score = root_mean_squared_error(y_va, oof_predictions[val_idx])
#     print(f"Fold {fold + 1} Optimized RMSLE: {fold_score:.4f}")

# cv_score = root_mean_squared_error(y_train, oof_predictions)
# print(f"\n======================================")
# print(f"Optimized Out-of-Fold CV RMSLE: {cv_score:.4f}")
# print(f"======================================")

# # Generate submission array ready for your Kaggle competition csv
# # Using expm1 to convert log-space back to real dollar values
# final_submissions = np.expm1(test_predictions)

In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

# Re-incorporate TargetValue into a temporary engineering dataframe
eda_train = X_train.copy()
eda_train['TargetValue_Log'] = y_train

# Initialize columns for augmented features
X_train['Model_Encoded_Median'] = np.nan
X_test['Model_Encoded_Median'] = np.nan

# Global median as fallback
global_median = eda_train['TargetValue_Log'].median()

# Out-of-Fold Target Encoding loop for Training Data
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, val_idx in kf.split(eda_train):
    # Compute median mapping using ONLY training folds
    fold_train = eda_train.iloc[train_idx]
    model_medians = fold_train.groupby('Spec_FullDescriptor')['TargetValue_Log'].median()
    
    # Map back to the validation fold
    X_train.iloc[val_idx, X_train.columns.get_loc('Model_Encoded_Median')] = \
        X_train.iloc[val_idx]['Spec_FullDescriptor'].map(model_medians)

# Fill missing values/unseen categories with global baseline
X_train['Model_Encoded_Median'] = X_train['Model_Encoded_Median'].fillna(global_median)

# For Test Data, map using the ENTIRE training set mapping
full_train_medians = eda_train.groupby('Spec_FullDescriptor')['TargetValue_Log'].median()
X_test['Model_Encoded_Median'] = X_test['Spec_FullDescriptor'].map(full_train_medians).fillna(global_median)

print("--- Target Encoding Augmentation Complete ---")
print(X_train['Model_Encoded_Median'].describe())

--- Target Encoding Augmentation Complete ---
count    138701.000000
mean         10.422889
std           0.594315
min           8.922792
25%           9.998843
50%          10.470224
75%          10.915107
max          11.819985
Name: Model_Encoded_Median, dtype: float64


/tmp/ipykernel_16/2292124381.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  model_medians = fold_train.groupby('Spec_FullDescriptor')['TargetValue_Log'].median()
/tmp/ipykernel_16/2292124381.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  model_medians = fold_train.groupby('Spec_FullDescriptor')['TargetValue_Log'].median()
/tmp/ipykernel_16/2292124381.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  model_

In [12]:
print("--- Interaction Analysis: Cabin Type vs Hydraulics (col10) ---")
interaction_table = pd.crosstab(
    train['CabinType'], 
    train['col10'].fillna('Missing'), 
    values=train['TargetValue'], 
    aggfunc='median'
)
print(interaction_table)

# Feature Augmentation: Combine them into a new compound feature
X_train['Cabin_Hydraulics_Combo'] = X_train['CabinType'].astype(str) + "_" + X_train['col10'].astype(str)
X_test['Cabin_Hydraulics_Combo'] = X_test['CabinType'].astype(str) + "_" + X_test['col10'].astype(str)

X_train['Cabin_Hydraulics_Combo'] = X_train['Cabin_Hydraulics_Combo'].astype('category')
X_test['Cabin_Hydraulics_Combo'] = X_test['Cabin_Hydraulics_Combo'].astype('category')

--- Interaction Analysis: Cabin Type vs Hydraulics (col10) ---
col10       2 Valve  3 Valve  4 Valve  Auxiliary  Base + 1 Function  \
CabinType                                                             
EROPS       34000.0  35000.0  52000.0    19000.0            31000.0   
EROPS w AC  55000.0  58000.0  68000.0    44000.0            75000.0   
OROPS       32000.0  37500.0  45000.0    16500.0            24000.0   

col10       Base + 2 Function  Base + 3 Function  Base + 4 Function  \
CabinType                                                             
EROPS                 60000.0            69000.0            88000.0   
EROPS w AC            91000.0            81000.0            92500.0   
OROPS                     NaN            77500.0                NaN   

col10       Base + 5 Function  Base + 6 Function  Missing  Standard  
CabinType                                                            
EROPS                 71750.0            50000.0  22000.0   26000.0  
EROPS w AC     

In [13]:
print("\n--- Missing Hours vs Price Distribution Analysis ---")
print("Median Price when OperationalHoursMeter is MISSING:")
print(train[train['OperationalHoursMeter'].isnull()]['TargetValue'].median())

print("\nMedian Price when OperationalHoursMeter is PRESENT:")
print(train[train['OperationalHoursMeter'].notnull()]['TargetValue'].median())


--- Missing Hours vs Price Distribution Analysis ---
Median Price when OperationalHoursMeter is MISSING:
36000.0

Median Price when OperationalHoursMeter is PRESENT:
35000.0


In [14]:
# 1. Component Extraction: Grab the last word/code from the full model descriptor
# This isolates the generation/series tag (e.g., 'G', 'HNA')
X_train['Model_Series_Tag'] = train['Spec_FullDescriptor'].fillna('Missing').apply(lambda x: x.split()[-1])
X_test['Model_Series_Tag'] = test['Spec_FullDescriptor'].fillna('Missing').apply(lambda x: x.split()[-1])

# 2. Frequency Encoding Augmentation
# High-cardinality values that appear rarely are highly volatile. Let's provide the model with the exact frequency count.
model_counts = train['Spec_FullDescriptor'].value_counts().to_dict()
X_train['Model_Frequency'] = train['Spec_FullDescriptor'].map(model_counts)
X_test['Model_Frequency'] = test['Spec_FullDescriptor'].map(model_counts).fillna(1) # Unseen items get a count of 1

# Convert string components back to clean categorical types
X_train['Model_Series_Tag'] = X_train['Model_Series_Tag'].astype('category')
X_test['Model_Series_Tag'] = X_test['Model_Series_Tag'].astype('category')

print("--- Advanced Text-Feature Augmentation Complete ---")
print(X_train[['Model_Series_Tag', 'Model_Frequency']].head())

--- Advanced Text-Feature Augmentation Complete ---
  Model_Series_Tag  Model_Frequency
0           950FII             1253
1             310G             1280
2           790ELC              391
3             416D              454
4           430HAG               20


In [15]:
print("--- 1. High-Level Region Pricing Variance ---")
# Check how many regions exist and the price spread between the highest and lowest market
region_stats = train.groupby('RegionCode')['TargetValue'].agg(['count', 'median', 'mean']).sort_values(by='median', ascending=False)
print("Top 5 Most Expensive Regions:")
print(region_stats.head(5))
print("\nTop 5 Cheapest Regions:")
print(region_stats.tail(5))

print("\n--- 2. Geographical Data Density Check ---")
# Check if there are rare regions in the test set that might cause volatility
print(f"Unique regions in Train: {train['RegionCode'].nunique()}")
print(f"Unique regions in Test:  {test['RegionCode'].nunique()}")

--- 1. High-Level Region Pricing Variance ---
Top 5 Most Expensive Regions:
              count   median          mean
RegionCode                                
Rhode Island     14  50450.0  52992.857143
Unspecified     586  48500.0  51705.631399
South Dakota    123  45000.0  54628.455285
Nevada         2730  41500.0  47533.589744
Alabama        3723  41000.0  47471.214343

Top 5 Cheapest Regions:
               count   median          mean
RegionCode                                 
Washington      6063  27500.0  33997.064160
New York        2111  26500.0  32789.057319
Pennsylvania    2406  26500.0  33694.368246
Puerto Rico       11  25000.0  33136.363636
Washington DC      2  22750.0  22750.000000

--- 2. Geographical Data Density Check ---
Unique regions in Train: 53
Unique regions in Test:  51


In [16]:
# 1. Calculate regional baseline statistics from train data
region_medians = train.groupby('RegionCode')['TargetValue'].median()
region_counts = train['RegionCode'].value_counts()

# 2. Build a mapping dictionary for tiers based on median price thresholds
def assign_region_tier(region):
    if region not in region_medians:
        return 'Mid_Tier' # Fallback for unseen test data
    median = region_medians[region]
    if median >= 40000:
        return 'Premium_Tier_Region'
    elif median >= 32000:
        return 'Standard_Tier_Region'
    else:
        return 'Value_Tier_Region'

# 3. Create Region Tier features
X_train['Region_Economic_Tier'] = train['RegionCode'].apply(assign_region_tier).astype('category')
X_test['Region_Economic_Tier'] = test['RegionCode'].apply(assign_region_tier).astype('category')

# 4. Create Low-Density Regional Risk Flag (Counts < 100 entries)
# This explicitly warns the model if a regional calculation lacks historical confidence
X_train['Is_Low_Density_Region'] = train['RegionCode'].map(region_counts).apply(lambda x: 1 if x < 100 else 0)
X_test['Is_Low_Density_Region'] = test['RegionCode'].map(region_counts).fillna(1).apply(lambda x: 1 if x < 100 else 0)

print("--- Geographical Feature Augmentation Complete ---")
print(X_train['Region_Economic_Tier'].value_counts())

--- Geographical Feature Augmentation Complete ---
Region_Economic_Tier
Standard_Tier_Region    113560
Value_Tier_Region        17965
Premium_Tier_Region       7176
Name: count, dtype: int64


In [17]:
# Create a temporary dataframe for deep cross-sectional analysis
eda_df = pd.DataFrame({
    'BaseClass': train['Spec_BaseClass'],
    'Age': X_train['EquipmentAgeAtSale'],
    'Price': train['TargetValue']
})

# Filter for the top 4 most common machine types to keep the comparison clean
top_4_classes = ['140', '320', 'D6', '12']
eda_filtered = eda_df[eda_df['BaseClass'].isin(top_4_classes)].dropna()

print("--- Median Price by Age Brackets for Dominant Machine Types ---")
# Group by both asset type and age categories to see the slopes
eda_filtered['Age_Group'] = pd.cut(eda_filtered['Age'], bins=[0, 5, 12, 25, 60], labels=['0-5 Yrs', '6-12 Yrs', '13-25 Yrs', '25+ Yrs'])
pivot_depreciation = eda_filtered.groupby(['BaseClass', 'Age_Group'], observed=False)['Price'].median().unstack(level=0)
print(pivot_depreciation)

--- Median Price by Age Brackets for Dominant Machine Types ---
BaseClass        12       140      320       D6
Age_Group                                      
0-5 Yrs    110000.0  112500.0  77500.0  92000.0
6-12 Yrs    76000.0   84000.0  51000.0  60000.0
13-25 Yrs   42000.0   55000.0  32500.0  42000.0
25+ Yrs     15500.0   43000.0  49250.0  22000.0


In [18]:
# 1. Update the function to look at the features present in the matrices
def assign_vintage_status(row):
    base_class = str(row['Spec_BaseClass'])
    age = row['EquipmentAgeAtSale']
    
    if pd.isna(age):
        return 'Unknown_Age'
        
    # Check for our resilient/rebuilt vintage anomalies over 25 years old
    if age >= 25:
        if base_class in ['320', '140']:
            return 'Premium_Vintage_Survivor'
        else:
            return 'Standard_Old_Machine'
    elif age <= 5:
        return 'Fresh_Asset'
    else:
        return 'Mid_Life_Depreciation'

# 2. We add 'Spec_BaseClass' temporarily to a tracking copy to process row-wise safely
temp_train = X_train.copy()
temp_train['Spec_BaseClass'] = train['Spec_BaseClass']

temp_test = X_test.copy()
temp_test['Spec_BaseClass'] = test['Spec_BaseClass']

# 3. Apply across the dataset copies
X_train['Asset_Lifecycle_Status'] = temp_train.apply(assign_vintage_status, axis=1).astype('category')
X_test['Asset_Lifecycle_Status'] = temp_test.apply(assign_vintage_status, axis=1).astype('category')

print("--- Asset Lifecycle Augmentation Complete ---")
print(X_train['Asset_Lifecycle_Status'].value_counts())

--- Asset Lifecycle Augmentation Complete ---
Asset_Lifecycle_Status
Mid_Life_Depreciation       78513
Fresh_Asset                 38221
Unknown_Age                 14747
Standard_Old_Machine         6603
Premium_Vintage_Survivor      617
Name: count, dtype: int64


In [19]:
print("--- Final Matrix Shape Evaluation ---")
print(f"Final Train Shape: {X_train.shape}")
print(f"Final Test Shape:  {X_test.shape}")

print("\n--- Summary of New Engineered Features ---")
new_features = [col for col in X_train.columns if col not in train.columns]
print(new_features)

--- Final Matrix Shape Evaluation ---
Final Train Shape: (138701, 61)
Final Test Shape:  (15000, 61)

--- Summary of New Engineered Features ---
['TransactionYear', 'TransactionMonth', 'TransactionDay', 'TransactionDayOfWeek', 'EquipmentAgeAtSale', 'Hours_Is_Missing', 'Hours_Per_Year', 'BaseClass_Mean_Hours', 'Relative_Usage_Index', 'Log_Equipment_Age', 'Model_Encoded_Median', 'Cabin_Hydraulics_Combo', 'Model_Series_Tag', 'Model_Frequency', 'Region_Economic_Tier', 'Is_Low_Density_Region', 'Asset_Lifecycle_Status']


In [20]:
import pandas as pd

# Combine continuous engineered features with the log target
engineered_numeric = [
    'EquipmentAgeAtSale', 'Hours_Per_Year', 'BaseClass_Mean_Hours', 
    'Relative_Usage_Index', 'Log_Equipment_Age', 'Model_Encoded_Median', 
    'Model_Frequency'
]

correlation_matrix = X_train[engineered_numeric].copy()
correlation_matrix['Log_TargetValue'] = y_train

# Calculate correlation with the target
rankings = correlation_matrix.corr()['Log_TargetValue'].sort_values(ascending=False)

print("--- Engineered Feature Correlation Leaderboard ---")
print(rankings)

--- Engineered Feature Correlation Leaderboard ---
Log_TargetValue         1.000000
Model_Encoded_Median    0.861649
Model_Frequency         0.238723
Hours_Per_Year          0.187498
BaseClass_Mean_Hours    0.153382
Relative_Usage_Index   -0.024953
EquipmentAgeAtSale     -0.375471
Log_Equipment_Age      -0.388286
Name: Log_TargetValue, dtype: float64


In [21]:
print("\n--- Final Null Value Profile (Engineered Columns Only) ---")
all_engineered = new_features # Using the list of 17 features verified in the last step
print(X_train[all_engineered].isnull().sum().sort_values(ascending=False))


--- Final Null Value Profile (Engineered Columns Only) ---
Hours_Per_Year            65853
Relative_Usage_Index      56643
EquipmentAgeAtSale        14747
Log_Equipment_Age         14747
BaseClass_Mean_Hours        619
TransactionYear               0
TransactionDayOfWeek          0
TransactionMonth              0
TransactionDay                0
Hours_Is_Missing              0
Model_Encoded_Median          0
Cabin_Hydraulics_Combo        0
Model_Series_Tag              0
Model_Frequency               0
Region_Economic_Tier          0
Is_Low_Density_Region         0
Asset_Lifecycle_Status        0
dtype: int64


In [22]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# Prepare a unified dataframe for global correlation analysis
full_eda = X_train.copy()
full_eda['Log_TargetValue'] = y_train

def cramers_v(x, y):
    """ Calculate Cramér's V for two categorical columns """
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))    
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

# Calculate association score of ALL columns directly with the Log_TargetValue
global_associations = {}

for col in full_eda.columns:
    if col == 'Log_TargetValue':
        continue
    try:
        if full_eda[col].dtype.name == 'category' or full_eda[col].dtype == 'object':
            # Categorical vs Numerical: Use group-by variance ratio
            grouped = full_eda.groupby(col, observed=True)['Log_TargetValue']
            global_mean = full_eda['Log_TargetValue'].mean()
            ss_total = ((full_eda['Log_TargetValue'] - global_mean)**2).sum()
            ss_between = grouped.apply(lambda x: len(x) * (x.mean() - global_mean)**2).sum()
            score = np.sqrt(ss_between / ss_total) if ss_total > 0 else 0
        else:
            # Numerical vs Numerical: Standard Pearson correlation (absolute value)
            score = abs(full_eda[col].corr(full_eda['Log_TargetValue']))
            
        global_associations[col] = score
    except Exception:
        continue

# Convert to sorted series
global_corr_leaderboard = pd.Series(global_associations).sort_values(ascending=False)

print("--- Top 25 Most Powerful Predictive Drivers Across Entire Dataset ---")
print(global_corr_leaderboard.head(50))

--- Top 25 Most Powerful Predictive Drivers Across Entire Dataset ---
Spec_FullDescriptor          0.876526
Model_Series_Tag             0.876466
Model_Encoded_Median         0.861649
Spec_BaseClass               0.720257
FunctionalClassification     0.638826
Cabin_Hydraulics_Combo       0.520173
AssetScaleFactor             0.515519
Spec_SubClass                0.469290
CabinType                    0.456179
Log_Equipment_Age            0.388286
Asset_Lifecycle_Status       0.383275
Spec_VariantModifier         0.377718
EquipmentAgeAtSale           0.375471
col20                        0.316481
ManufactureYear              0.296555
Spec_ReleaseSeries           0.261019
col10                        0.256562
Model_Frequency              0.238723
InventoryGroupDescription    0.230653
InventoryGroupCategory       0.230653
col22                        0.222269
col15                        0.217265
col3                         0.210968
col12                        0.205308
UtilizationTier   

In [23]:
# 1. Feature Augmentation: Out-of-Fold Target Encoding for FunctionalClassification
X_train['Func_Class_Encoded_Median'] = np.nan
X_test['Func_Class_Encoded_Median'] = np.nan

# Global fallback
global_median_log = y_train.median()

# Re-use KFold for leak-free train mapping
for train_idx, val_idx in kf.split(X_train):
    fold_train_target = y_train.iloc[train_idx]
    fold_train_feature = train.iloc[train_idx]
    
    class_medians = fold_train_target.groupby(fold_train_feature['FunctionalClassification']).median()
    X_train.iloc[val_idx, X_train.columns.get_loc('Func_Class_Encoded_Median')] = \
        train.iloc[val_idx]['FunctionalClassification'].map(class_medians)

X_train['Func_Class_Encoded_Median'] = X_train['Func_Class_Encoded_Median'].fillna(global_median_log)

# Map to Test set using full train data rules
full_class_medians = y_train.groupby(train['FunctionalClassification']).median()
X_test['Func_Class_Encoded_Median'] = test['FunctionalClassification'].map(full_class_medians).fillna(global_median_log)


# 2. Prune Noise Features: Drop columns with weak correlation (< 0.10)
weak_cols = ['DataOriginCode', 'Forks', 'col14', 'col16', 'col23']
X_train = X_train.drop(columns=[col for col in weak_cols if col in X_train.columns])
X_test = X_test.drop(columns=[col for col in weak_cols if col in X_test.columns])


print("--- Correlation-Based Optimization Complete ---")
print(f"Final Train Shape: {X_train.shape}")
print(f"Final Test Shape:  {X_test.shape}")

--- Correlation-Based Optimization Complete ---
Final Train Shape: (138701, 57)
Final Test Shape:  (15000, 57)


In [24]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

# Initialize K-Fold identically to keep comparison clean
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_predictions = np.zeros(len(X_train))
test_predictions = np.zeros(len(X_test))

print("--- Training Model on Optimized 57-Feature Dataset ---")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Using our reliable baseline parameters to isolate data quality improvements
    model = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        n_estimators=1500,
        learning_rate=0.05,
        num_leaves=63,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    oof_predictions[val_idx] = model.predict(X_va)
    test_predictions += model.predict(X_test) / kf.n_splits
    
    fold_score = root_mean_squared_error(y_va, oof_predictions[val_idx])
    print(f"Fold {fold + 1} RMSLE: {fold_score:.4f}")

cv_score = root_mean_squared_error(y_train, oof_predictions)
print(f"\n======================================")
print(f"Final Optimized Data CV RMSLE Score: {cv_score:.4f}")
print(f"======================================")

# Convert log predictions back to regular currency values for your submission
final_kaggle_predictions = np.expm1(test_predictions)


--- Training Model on Optimized 57-Feature Dataset ---
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017918 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7753
[LightGBM] [Info] Number of data points in the train set: 110960, number of used features: 57
[LightGBM] [Info] Start training from score 10.424734
Fold 1 RMSLE: 0.2032
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threadi

In [25]:
import lightgbm as lgb
from catboost import CatBoostRegressor
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

# Identify remaining categorical columns for models that need explicit declaration
cat_cols = [col for col in X_train.columns if X_train[col].dtype.name == 'category']

# Convert categories to integer codes ONLY for XGBoost (as it requires numeric matrices)
X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()
for col in cat_cols:
    X_train_xgb[col] = X_train_xgb[col].cat.codes
    X_test_xgb[col] = X_test_xgb[col].cat.codes

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Matrices to store cross-validation predictions
oof_lgb = np.zeros(len(X_train))
oof_cat = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))

test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))

print("--- Starting Multi-Model Ensembling Pipeline ---")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    print(f"\n--- Training Fold {fold + 1} ---")
    
    # Split data sets
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    X_tr_xgb, X_va_xgb = X_train_xgb.iloc[train_idx], X_train_xgb.iloc[val_idx]

    # 1. LIGHTGBM
    model_lgb = lgb.LGBMRegressor(objective='regression', metric='rmse', n_estimators=1500, learning_rate=0.05, num_leaves=63, random_state=42, n_jobs=-1, verbose=-1)
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = model_lgb.predict(X_va)
    test_lgb += model_lgb.predict(X_test) / kf.n_splits

    # 2. CATBOOST (Symmetric Trees - incredible with high-cardinality strings)
    model_cat = CatBoostRegressor(iterations=1500, learning_rate=0.05, depth=6, loss_function='RMSE', random_seed=42, thread_count=-1, verbose=False)
    model_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=cat_cols, early_stopping_rounds=50)
    oof_cat[val_idx] = model_cat.predict(X_va)
    test_cat += model_cat.predict(X_test) / kf.n_splits

    # 3. XGBOOST (Pre-sorted/Histogram-based depthwise splits)
    model_xgb = xgb.XGBRegressor(n_estimators=1500, learning_rate=0.05, max_depth=6, tree_method='hist', random_state=42, n_jobs=-1)
    model_xgb.fit(X_tr_xgb, y_tr, eval_set=[(X_va_xgb, y_va)], verbose=False) # early stopping omitted for basic script compatibility
    oof_xgb[val_idx] = model_xgb.predict(X_va_xgb)
    test_xgb += model_xgb.predict(X_test_xgb) / kf.n_splits

print("\n--- Final CV Score Evaluation Leaderboard ---")
print(f"LightGBM OOF RMSLE: {root_mean_squared_error(y_train, oof_lgb):.4f}")
print(f"CatBoost OOF RMSLE: {root_mean_squared_error(y_train, oof_cat):.4f}")
print(f"XGBoost  OOF RMSLE: {root_mean_squared_error(y_train, oof_xgb):.4f}")

--- Starting Multi-Model Ensembling Pipeline ---

--- Training Fold 1 ---

--- Training Fold 2 ---

--- Training Fold 3 ---

--- Training Fold 4 ---

--- Training Fold 5 ---

--- Final CV Score Evaluation Leaderboard ---
LightGBM OOF RMSLE: 0.2050
CatBoost OOF RMSLE: 0.2161
XGBoost  OOF RMSLE: 0.2091


In [26]:
# Blending the test arrays (Adjust weights dynamically based on which model scores best!)
# This assumes LightGBM and CatBoost will outperform XGBoost on this categorical-heavy data
final_blend_log_predictions = (0.40 * test_lgb) + (0.40 * test_cat) + (0.20 * test_xgb)

# Inverse transform out of log-space back to real dollar valuations
final_prices = np.expm1(final_blend_log_predictions)

# Create submission file
sub = pd.read_csv('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv')
sub['TargetValue'] = final_prices
sub.to_csv('submission.csv', index=False)

print("\n--- Blended Submission File Created! ---")


--- Blended Submission File Created! ---
